# Week 26, Day 6: Part 1 exam

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w026/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: B1: how many latent signals?

80 neurons are driven by two latent signals plus independent noise of sd 0.5. Find the covariance eigenvalues `lam` (descending) and eigenvectors, count how many exceed the largest eigenvalue pure noise would produce, edge = σ²(1 + √(p/n))², in `n_signal`, then regress the latents on the top two component scores and store each latent's R² in `r2_latent`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(41); T, p = 500, 80; t = np.arange(T) * 0.01
lat = np.c_[np.sin(2 * np.pi * 0.5 * t), np.cos(2 * np.pi * 0.3 * t)]
Y = lat @ r.normal(size=(2, p)) + 0.5 * r.normal(size=(T, p))
Yc = Y - Y.mean(0)
lam, V = ...                                      # eigen-decomposition of the covariance, sorted descending
edge = 0.5 ** 2 * (1 + np.sqrt(p / T)) ** 2
n_signal = ...
Z = Yc @ V[:, :2]
L = lat - lat.mean(0)
B = np.linalg.lstsq(Z, L, rcond=None)[0]
r2_latent = ...
print("top eigenvalues", lam[:5].round(2), " noise edge", round(edge, 2), " signals", n_signal, " R2", np.round(r2_latent, 3))

In [ ]:
import numpy as np
def test_eigs():
    "lam holds the covariance eigenvalues in descending order"
    ref = np.sort(np.linalg.eigvalsh(np.cov(Y, rowvar=False)))[::-1]
    assert np.allclose(lam, ref)

def test_count():
    "n_signal counts eigenvalues above the noise edge, and equals the true number of latents"
    assert n_signal == int(np.sum(lam > edge)) == 2

def test_recover():
    "the top two components predict both latent signals with R squared above 0.95"
    assert np.all(r2_latent > 0.95)

test_eigs()
test_count()
test_recover()
print("All checks passed")

### Exercise: B2: a Poisson GLM by Newton's method

Spike counts y depend on three stimulus features through a Poisson GLM, y ~ Poisson(exp(Xw)). Maximise the log likelihood Σ(y Xw − exp(Xw)) with Newton's method: gradient Xᵀ(y − μ), Hessian of the negative log likelihood Xᵀ diag(μ) X. Iterate 20 times from zero to get `w_hat`, and take standard errors `se` from the inverse Hessian at the solution.

In [ ]:
r = np.random.default_rng(31); n = 2000
X = np.c_[np.ones(n), r.normal(size=(n, 3))]; w_true = np.array([0.5, 0.8, -0.4, 0.0])
y = r.poisson(np.exp(X @ w_true))
w = np.zeros(4)
for _ in range(20):
    mu = np.exp(X @ w)
    g = ...
    H = ...
    w = w + np.linalg.solve(H, g)
w_hat = w
se = ...
print("estimate", w_hat.round(3), " truth", w_true, " SE", se.round(3))

In [ ]:
import numpy as np
def test_grad():
    "the gradient is zero at the solution"
    mu = np.exp(X @ w_hat)
    assert np.abs(X.T @ (y - mu)).max() < 1e-6

def test_close():
    "every weight is within three standard errors of the truth"
    assert np.all(np.abs(w_hat - w_true) < 3 * se)

test_grad()
test_close()
print("All checks passed")

### Exercise: B3: three ways to filter a spike train

Filter a spike train with an exponential kernel (τ = 20 ms) three ways: direct convolution (`direct`), convolution through the FFT with zero padding to length T + K − 1 (`via_fft`), and by Euler integration of dv/dt = −v/τ + spikes (`v`). Show they agree.

In [ ]:
r = np.random.default_rng(51); dt = 0.001; T = 2000
spikes = (r.random(T) < 0.02).astype(float) / dt     # spike train as a sum of deltas
tau = 0.02; kernel = np.exp(-np.arange(0, 0.2, dt) / tau)
direct = np.convolve(spikes, kernel)[:T] * dt
m = T + len(kernel) - 1
via_fft = ...
v = np.zeros(T)
for i in range(1, T):
    v[i] = ...
print(f"FFT vs direct: {np.abs(via_fft - direct).max():.1e};  Euler vs convolution: {np.abs(v[1:] - direct[:-1]).max() / direct.max():.3f} of the peak")

In [ ]:
import numpy as np
def test_fft():
    "the FFT convolution equals direct convolution to 1e-9"
    assert np.abs(via_fft - direct).max() < 1e-9

def test_ode():
    "the Euler solution of the leaky integrator matches the convolution within 5% of its peak"
    assert np.abs(v[1:] - direct[:-1]).max() / direct.max() < 0.05

test_fft()
test_ode()
print("All checks passed")

### Exercise: B4: a three-state channel

A channel moves C1 ⇄ C2 ⇄ O with rates C1→C2 0.5, C2→C1 0.2, C2→O 1.0, O→C2 0.8 per ms. Build the generator Q (rows sum to zero), find the stationary distribution `pi` as the null vector of Qᵀ, then simulate 40,000 dwells with the Gillespie algorithm (local rng seed 61) and store the fraction of time in each state in `frac` and the mean open dwell time in `mean_open`.

In [ ]:
Q = np.array([[-0.5, 0.5, 0.0], [0.2, -1.2, 1.0], [0.0, 0.8, -0.8]])
w_, Vq = np.linalg.eig(Q.T)
pi = ...
rg = np.random.default_rng(61); s = 0; occ = np.zeros(3); opens = []
for _ in range(40000):
    d = ...                                       # exponential dwell with rate -Q[s, s]
    occ[s] += d
    if s == 2: opens.append(d)
    p_next = Q[s].copy(); p_next[s] = 0
    s = rg.choice(3, p=p_next / p_next.sum())
frac = occ / occ.sum(); mean_open = np.mean(opens)
print("stationary", pi.round(4), " simulated", frac.round(4), f" mean open {mean_open:.3f} ms")

In [ ]:
import numpy as np
def test_pi():
    "pi solves pi Q = 0 and sums to 1"
    assert abs(pi.sum() - 1) < 1e-12 and np.abs(pi @ Q).max() < 1e-12

def test_sim():
    "the Gillespie fractions of time match pi within 0.02 and the mean open time is within 5% of 1/0.8"
    assert np.abs(frac - pi).max() < 0.02 and abs(mean_open - 1.25) / 1.25 < 0.05

test_pi()
test_sim()
print("All checks passed")